# Multi-Object Player Tracking & Dataset Automation (YOLO11x + ByteTrack)

End-to-end automated tracking pipeline across all 45 broadcast basketball sequences in the SportsMOT dataset (`train`).

### Key Pipeline Capabilities
* **High-Resolution Detection & Tracking:** `YOLO11x` (`imgsz=1280`, `conf=0.25`) paired with `ByteTrack` (`persist=True`) to maintain consistent player identities across fast motion and severe paint occlusions.
* **Per-Sequence Tracker Isolation:** Re-initializes the tracker per clip to prevent cross-sequence ID bleeding.
* **Geometric Court Masking:** Filters out referees, bench staff, and spectators using foot-point polygon containment (`cv2.pointPolygonTest`).
* **Downstream Handoff Deliverables:**
  * `all_sequences_tracks_filtered.csv` — Master frame-by-frame tracking dataset (`track_id`, bounding boxes, and `feet_x`/`feet_y` floor contact coordinates) for Court Homography and Pose Estimation.
  * `dataset_evaluation_summary.csv` — Clip-by-clip Precision, Recall, F1-Score, and unique track counts benchmarked against ground truth.
  * `tracking_demos/` — Annotated `.mp4` tracking videos with persistent IDs and 25-frame player motion trajectories.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
    # for filename in filenames:
        # print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 1. Environment Setup & Hardware Verification

Installs `ultralytics` and verifies NVIDIA CUDA GPU availability for accelerated FP16 inference.

In [2]:
!pip install -q ultralytics
import torch

print(
    f"CUDA Available: {torch.cuda.is_available()} | Device:"
    f" {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}"
)

CUDA Available: True | Device: Tesla T4


In [3]:
from pathlib import Path

for base in [Path("/kaggle/input"), Path("/kaggle/working")]:
  print(f"=== {base} ===")
  for p in sorted(base.rglob("*"))[:30]:
    print(" ", p)


print("Files in /kaggle/working:", list(Path("/kaggle/working").glob("*.csv")))

=== /kaggle/input ===
  /kaggle/input/datasets
  /kaggle/input/datasets/satyabratadas10
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/._train
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_-6Os86HzwCs_c001
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_-6Os86HzwCs_c003
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_-6Os86HzwCs_c007
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_-6Os86HzwCs_c009
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_1LwtoLPw2TU_c006
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_1LwtoLPw2TU_c012
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_1LwtoLPw2TU_c014
  /kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train/._v_1LwtoLPw2TU_c016
  /kaggle/i

## 2. Full-Dataset Multi-Object Tracking & Quantitative Evaluation

Streams all 45 sequences from the SportsMOT dataset, executes frame-by-frame player tracking with spatial court filtering, evaluates detections against ground truth (`IoU >= 0.5`), and exports the master handoff CSVs.

### CELL A: Build a single-class "player" YOLO dataset from the SportsMOT GT

In [9]:
from pathlib import Path
import random
import shutil

import numpy as np
from PIL import Image

dataset_root = Path(
    "/kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train"
)
yolo_root = Path("/kaggle/working/sportsmot_players_yolo")
FRAME_STRIDE = 5      # neighbouring frames are almost identical; keep every 5th
VAL_FRACTION = 0.2    # fraction of *source videos* held out for validation

seq_dirs = sorted(
    d for d in dataset_root.iterdir()
    if d.is_dir() and not d.name.startswith("._")
    and (d / "img1").exists() and (d / "gt" / "gt.txt").exists()
)

# Split by source video (e.g. "v_-6Os86HzwCs"), not by clip. Clips from the
# same game look almost the same, so splitting by clip would leak into val.
def video_id(seq_name):
    return seq_name.rsplit("_c", 1)[0]

videos = sorted({video_id(d.name) for d in seq_dirs})
random.Random(0).shuffle(videos)
val_videos = set(videos[: max(1, int(len(videos) * VAL_FRACTION))])

n_imgs = {"train": 0, "val": 0}
for seq_dir in seq_dirs:
    split = "val" if video_id(seq_dir.name) in val_videos else "train"
    img_out = yolo_root / "images" / split
    lbl_out = yolo_root / "labels" / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    # gt.txt columns: frame, id, x, y, w, h, ...
    gt = np.loadtxt(seq_dir / "gt" / "gt.txt", delimiter=",", ndmin=2)
    img_paths = sorted(
        p for p in (seq_dir / "img1").glob("*.jpg") if not p.name.startswith("._")
    )
    W, H = Image.open(img_paths[0]).size

    for img_path in img_paths[::FRAME_STRIDE]:
        frame_num = int(img_path.stem)
        rows = gt[gt[:, 0] == frame_num]
        lines = []
        for _, _, x, y, w, h, *_ in rows:
            x1, y1 = max(0.0, x), max(0.0, y)
            x2, y2 = min(W, x + w), min(H, y + h)
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
            bw, bh = (x2 - x1) / W, (y2 - y1) / H
            lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        stem = f"{seq_dir.name}_{frame_num:06d}"
        shutil.copy(img_path, img_out / f"{stem}.jpg")
        (lbl_out / f"{stem}.txt").write_text("\n".join(lines))
        n_imgs[split] += 1

(yolo_root / "data.yaml").write_text(
    f"path: {yolo_root}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: player\n"
)
print(f"Dataset ready: {n_imgs} | val videos: {sorted(val_videos)}")

Dataset ready: {'train': 5399, 'val': 331} | val videos: ['v_1LwtoLPw2TU', 'v_dChHNGIfm4Y']


### Fine-tune YOLO on player-only labels

In [10]:
from ultralytics import YOLO

det_model = YOLO("yolo11s.pt")
det_model.train(
    data=str(yolo_root / "data.yaml"),
    imgsz=1280,
    epochs=25,
    batch=-1,            # auto-batch to fit T4 memory
    device=0,
    workers=2,
    patience=8,
    close_mosaic=5,
    project="/kaggle/working/runs",
    name="player_det",
    exist_ok=True,
)
PLAYER_WEIGHTS = "/kaggle/working/runs/player_det/weights/best.pt"

Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=5, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/sportsmot_players_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=player_det, nbs=64, nms=None, opset=None

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/25      7.62G     0.8321     0.5254     0.9861         22       1280: 100% ━━━━━━━━━━━━ 772/772 2.8it/s 4:370.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 5.2it/s 4.6s0.2s
                   all        331       3838      0.936      0.933      0.971      0.795

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/25      7.55G     0.8162     0.5076     0.9765         15       1280: 100% ━━━━━━━━━━━━ 772/772 2.8it/s 4:370.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 5.1it/s 4.7s0.2s
                   all        331       3838      0.947      0.913      0.975      0.802

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/25      7.58G     0.7917     0.4799     0.9685         16       1280: 100% ━━━━━━━━━━━━ 772/772 2.8it/s 4:370.2ss
                 Class     Images

### Track with the player-only detector, clean fragments, evaluate the
- FINAL output (after filtering) against GT, export CSVs

In [11]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

PLAYER_WEIGHTS = "/kaggle/working/runs/player_det/weights/best.pt"
dataset_root = Path(
    "/kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train"
)
max_sequences = 3        # None = all clips
ONLY_VAL_CLIPS = False   # True = honest metrics on clips the model never saw
CONF = 0.35
IMGSZ = 1280
MIN_TRACK_LEN = 10       # drop tracks shorter than this many frames (flicker / fragments)

# ByteTrack with a longer memory so occluded players keep their ID instead of
# spawning a new one (fewer IDs = fewer fragments to misclassify).
tracker_cfg = Path("/kaggle/working/bytetrack_players.yaml")
tracker_cfg.write_text(
    "tracker_type: bytetrack\n"
    "track_high_thresh: 0.4\n"
    "track_low_thresh: 0.1\n"
    "new_track_thresh: 0.5\n"
    "track_buffer: 60\n"
    "match_thresh: 0.8\n"
    "fuse_score: True\n"
)


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_frame(pred_boxes, gt_boxes, thr=0.5):
    pairs = sorted(
        ((iou(p, g), i, j) for i, p in enumerate(pred_boxes) for j, g in enumerate(gt_boxes)),
        reverse=True,
    )
    used_p, used_g, tp = set(), set(), 0
    for s, i, j in pairs:
        if s < thr:
            break
        if i in used_p or j in used_g:
            continue
        used_p.add(i); used_g.add(j); tp += 1
    return tp, len(pred_boxes) - tp, len(gt_boxes) - tp


seq_dirs = sorted(
    d for d in dataset_root.iterdir()
    if d.is_dir() and not d.name.startswith("._") and (d / "img1").exists()
)
if ONLY_VAL_CLIPS:
    seq_dirs = [d for d in seq_dirs if d.name.rsplit("_c", 1)[0] in val_videos]
target_dirs = seq_dirs[:max_sequences] if max_sequences else seq_dirs

COLS = ["sequence", "frame", "track_id", "x1", "y1", "x2", "y2",
        "w", "h", "conf", "feet_x", "feet_y"]
all_dfs, summaries = [], []

for seq_idx, seq_dir in enumerate(target_dirs, start=1):
    seq_name = seq_dir.name
    img_paths = sorted(
        p for p in (seq_dir / "img1").glob("*.jpg") if not p.name.startswith("._")
    )

    model = YOLO(PLAYER_WEIGHTS)   # fresh model = fresh tracker state per clip
    records = []
    for img_path in img_paths:
        frame_num = int(img_path.stem)
        frame_bgr = cv2.imread(str(img_path))   # Ultralytics expects BGR arrays
        res = model.track(
            frame_bgr, persist=True, classes=[0], conf=CONF, imgsz=IMGSZ,
            device=0, tracker=str(tracker_cfg), verbose=False,
        )[0]
        if res.boxes.id is None:
            continue
        for box, tid, conf in zip(
            res.boxes.xyxy.cpu().numpy(),
            res.boxes.id.cpu().numpy().astype(int),
            res.boxes.conf.cpu().numpy(),
        ):
            x1, y1, x2, y2 = map(float, box)
            w, h = x2 - x1, y2 - y1
            records.append([
                seq_name, frame_num, int(tid),
                round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2),
                round(w, 2), round(h, 2), round(float(conf), 3),
                round(x1 + w / 2.0, 2), round(y2, 2),
            ])

    seq_df = pd.DataFrame(records, columns=COLS)
    n_raw_ids = seq_df["track_id"].nunique()

    # Remove short-lived tracks (flickers, ID fragments)
    track_len = seq_df.groupby("track_id")["frame"].transform("size")
    seq_df = seq_df[track_len >= MIN_TRACK_LEN].copy()
    all_dfs.append(seq_df)

    # Evaluate exactly what we export
    gt_path = seq_dir / "gt" / "gt.txt"
    prec = rec = f1 = float("nan")
    if gt_path.exists():
        gt = np.loadtxt(gt_path, delimiter=",", ndmin=2)
        preds = {f: g[["x1", "y1", "x2", "y2"]].values.tolist()
                 for f, g in seq_df.groupby("frame")}
        TP = FP = FN = 0
        for img_path in img_paths:
            f = int(img_path.stem)
            f_gt = gt[gt[:, 0] == f]
            gt_boxes = [(g[2], g[3], g[2] + g[4], g[3] + g[5]) for g in f_gt]
            tp, fp, fn = match_frame(preds.get(f, []), gt_boxes)
            TP += tp; FP += fp; FN += fn
        prec = TP / (TP + FP) if TP + FP else 0.0
        rec = TP / (TP + FN) if TP + FN else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0

    summaries.append({
        "Sequence": seq_name, "Frames": len(img_paths),
        "Precision": round(prec, 3), "Recall": round(rec, 3), "F1-Score": round(f1, 3),
        "Raw Tracks": n_raw_ids, "Kept Tracks": seq_df["track_id"].nunique(),
    })
    print(f"  [{seq_idx}/{len(target_dirs)}] {seq_name} | P {prec:.3f} | R {rec:.3f} "
          f"| tracks {n_raw_ids} -> {seq_df['track_id'].nunique()}")

df_players = pd.concat(all_dfs, ignore_index=True)
df_players.to_csv("/kaggle/working/all_sequences_players_only.csv", index=False)
df_summary = pd.DataFrame(summaries)
df_summary.to_csv("/kaggle/working/dataset_evaluation_summary.csv", index=False)
display(df_summary)

  [1/3] v_-6Os86HzwCs_c001 | P 0.987 | R 0.972 | tracks 19 -> 15
  [2/3] v_-6Os86HzwCs_c003 | P 0.991 | R 0.979 | tracks 20 -> 15
  [3/3] v_-6Os86HzwCs_c007 | P 0.993 | R 0.970 | tracks 16 -> 15


,Sequence,Frames,Precision,Recall,F1-Score,Raw Tracks,Kept Tracks
0,v_-6Os86HzwCs_c001,825,0.987,0.972,0.980,19,15
1,v_-6Os86HzwCs_c003,403,0.991,0.979,0.985,20,15
2,v_-6Os86HzwCs_c007,550,0.993,0.970,0.982,16,15


## 3. Multi-Object Tracking Video Demo Generation

Renders `.mp4` visual verification demos directly from `all_sequences_tracks_filtered.csv`, featuring color-coded player bounding boxes, persistent `Player ID` badges, 25-frame foot-point trajectory trails, and the active court boundary overlay.

### Render demo videos from all_sequences_players_only.csv
- (replaces cells 7, 8-part-3 and 10; cell 10 crashed because it read
- all_sequences_tracks_filtered.csv, which is never written)

In [12]:
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

df_tracks = pd.read_csv("/kaggle/working/all_sequences_players_only.csv")
dataset_root = Path(
    "/kaggle/input/datasets/satyabratadas10/sportsmot-train-tar/train"
)
demo_dir = Path("/kaggle/working/tracking_demos")
demo_dir.mkdir(exist_ok=True)


def get_id_color(track_id):
    rng = np.random.default_rng(int(track_id) * 42)
    c = rng.integers(60, 255, size=3)
    return int(c[0]), int(c[1]), int(c[2])


for seq_name in df_tracks["sequence"].unique()[:3]:
    img_paths = sorted(
        p for p in (dataset_root / seq_name / "img1").glob("*.jpg")
        if not p.name.startswith("._")
    )
    seq_df = df_tracks[df_tracks["sequence"] == seq_name]
    by_frame = {f: g for f, g in seq_df.groupby("frame")}
    H, W = cv2.imread(str(img_paths[0])).shape[:2]

    out_path = demo_dir / f"demo_players_only_{seq_name}.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), 25.0, (W, H))
    history = defaultdict(list)

    for img_path in img_paths:
        frame_num = int(img_path.stem)
        canvas = cv2.imread(str(img_path))
        rows = by_frame.get(frame_num, seq_df.iloc[0:0])
        for r in rows.itertuples():
            tid = int(r.track_id)
            color = get_id_color(tid)
            x1, y1, x2, y2 = int(r.x1), int(r.y1), int(r.x2), int(r.y2)

            history[tid].append((int(r.feet_x), int(r.feet_y)))
            history[tid] = history[tid][-25:]
            if len(history[tid]) > 1:
                cv2.polylines(canvas, [np.array(history[tid], np.int32).reshape(-1, 1, 2)],
                              False, color, 2)

            cv2.rectangle(canvas, (x1, y1), (x2, y2), color, 2)
            label = f"Player ID {tid}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(canvas, (x1, max(y1 - th - 8, 0)), (x1 + tw + 6, y1), color, -1)
            cv2.putText(canvas, label, (x1 + 3, max(y1 - 5, 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 2)

        cv2.rectangle(canvas, (15, 15), (520, 55), (30, 30, 30), -1)
        cv2.putText(canvas, f"Players Only | Frame: {frame_num}/{len(img_paths)} | Active: {len(rows)}",
                    (25, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2)
        writer.write(canvas)

    writer.release()
    print(f"Saved demo video: {out_path}")

Saved demo video: /kaggle/working/tracking_demos/demo_players_only_v_-6Os86HzwCs_c001.mp4
Saved demo video: /kaggle/working/tracking_demos/demo_players_only_v_-6Os86HzwCs_c003.mp4
Saved demo video: /kaggle/working/tracking_demos/demo_players_only_v_-6Os86HzwCs_c007.mp4
